# **Problem Statement**

## Business Context


In hazardous workplaces like construction sites and industrial plants, ensuring worker safety is critical. Head injuries caused by falling objects or accidents are among the leading causes of fatalities. Safety helmets are essential protective equipment, yet compliance with helmet regulations is often inconsistent, particularly in large-scale operations where manual monitoring is inefficient and prone to errors.

SafeGuard Corp aims to address this issue by automating safety monitoring through an advanced image analysis system. By detecting workers and identifying whether they are wearing helmets, this system will enhance compliance, minimize workplace injuries, and reduce human oversight errors.

## Objective

Given the challenges faced by SafeGuard Corp in ensuring helmet compliance at hazardous workplaces, they have hired you as a Data Scientist to develop an advanced, machine learning-based solution that achieves the following:

1. Utilize object detection techniques to accurately identify and locate workers in images captured from construction sites and industrial plants.

2. Implement a classification model that distinguishes whether the detected workers are wearing helmets or not.

3. Analyze patterns in the collected image data to understand the factors influencing helmet compliance and the common scenarios where lapses occur.

4. Integrate the system with existing safety protocols to provide real-time alerts and reports to safety officers, enabling prompt corrective actions.

5. Ensure the solution is scalable to handle increased volumes of image data from multiple sites, while maintaining high accuracy and efficiency.

This automated system aims to enhance compliance, reduce the risk of head injuries, and streamline the monitoring process, ultimately leading to a safer workplace environment.

## Data Description

The dataset consists of 640 images, equally divided into two categories:
1. WithHelmet: 320images showing workers wearing helmets.
2. WithoutHelmet: 320images showing workers not wearing helmets.

**Dataset Characteristics:**
1. Variations in Conditions: Images include diverse environments such as construction sites, factories, and industrial settings, with variations in lighting, angles, and worker postures to simulate real-world conditions.
2. WorkerActivities: Workers are depicted in different actions such as standing, using tools, or moving, ensuring robust model learning for various scenarios.

# **Please read the instructions carefully before starting the project.**
This is a commented Jupyter IPython Notebook file in which all the instructions and tasks to be performed are mentioned.
* Blanks '_______' are provided in the notebook that
needs to be filled with an appropriate code to get the correct result. With every '_______' blank, there is a comment that briefly describes what needs to be filled in the blank space.
* Identify the task to be performed correctly, and only then proceed to write the required code.
* Fill the code wherever asked by the commented lines like "# write your code here" or "# complete the code". Running incomplete code may throw error.
* Please run the codes in a sequential manner from the beginning to avoid any unnecessary errors.
* Add the results/observations (wherever mentioned) derived from the analysis in the presentation and submit the same.

# **Importing Necessary Libraries**

In [ ]:
#import nescessary libraries
import cv2 as cv
import torch
from PIL import Image, ImageDraw
import tensorflow as tf
import os
import random
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import random
import numpy as np
from sklearn.metrics import precision_score, recall_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
from sklearn.model_selection import train_test_split
import pandas as pd
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# **Loading the Data**

In [ ]:
# The dataset is already in the directory
extract_to_path = '/Users/muratgultekin/Projects/GREATLEARN/HELMET'
print(f'Using dataset from {extract_to_path}')


In [ ]:
training_data_images_path = extract_to_path + '/HelmetDetectionDataset/Training/'
validation_data_images_path = extract_to_path + '/HelmetDetectionDataset/Validation/'

# **Exploratory Data Analysis**


1. **Class Distribution Analysis**: This step involves examining how the different classes are distributed across the dataset. For instance, in a helmet detection task, you would check the number of images with people wearing helmets versus those without. This helps identify if there are any imbalances that might affect model performance.

2. **Visualizing Sample Images**: In this phase, we take a closer look at some of the images from the dataset. This could include displaying random samples or specific examples from each class. Visualizations help to understand the quality, diversity, and challenges in the data (e.g., lighting, posture, background, etc.), which is important for choosing the right model and preprocessing techniques.

In [ ]:
training_images_with_helmet = len(os.listdir(os.path.join(training_data_images_path, 'WorkersWithHelmet')))
training_images_without_helmet = len(os.listdir(os.path.join(training_data_images_path, 'WorkersWithoutHelmet')))
validation_images_with_helmet = len(os.listdir(os.path.join(validation_data_images_path, 'WorkersWithHelmet')))
validation_images_without_helmet = len(os.listdir(os.path.join(validation_data_images_path, 'WorkersWithoutHelmet')))

categories = ['Train With Helmet', 'Train Without Helmet', 'Validation With Helmet', 'Validation Without Helmet']
values = [training_images_with_helmet, training_images_without_helmet, validation_images_with_helmet, validation_images_without_helmet]


In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(categories, values, color=['green', 'red', 'blue', 'orange'])

# Adding titles and labels
plt.title('Distribution of Training and Validation Images', fontsize=14)
plt.xlabel('Categories', fontsize=12)
plt.ylabel('Number of Images', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Show the plot
plt.tight_layout()
plt.show()


Let's view some random images from the training dataset.

In [ ]:
# Function to plot random images
def plot_random_images(folder_path, title, num_images=5):
    # Get all image file names from the folder
    image_files = os.listdir(folder_path)
    # Randomly select the specified number of images
    selected_images = random.sample(image_files, num_images)

    # Plot the images
    plt.figure(figsize=(15, 5))
    for i, image_file in enumerate(selected_images):
        image_path = os.path.join(folder_path, image_file)
        img = mpimg.imread(image_path)
        plt.subplot(1, num_images, i + 1)  # Arrange images in 1 row
        plt.imshow(img)
        plt.axis('off')  # Hide axes
        plt.title(f'{title} #{i + 1}', fontsize=10)

    plt.tight_layout()
    plt.show()

In [ ]:
# Paths to the folders
with_helmet_path_training = os.path.join(training_data_images_path, 'WorkersWithHelmet')
without_helmet_path_training = os.path.join(training_data_images_path, 'WorkersWithoutHelmet')
with_helmet_path_validation = os.path.join(validation_data_images_path, 'WorkersWithHelmet')
without_helmet_path_validation = os.path.join(validation_data_images_path, 'WorkersWithoutHelmet')

In [ ]:
# Plot 5 random images with helmet
plot_random_images(with_helmet_path_training, 'With Helmet', num_images=5)


In [ ]:
# Plot 5 random images without helmet
plot_random_images(without_helmet_path_training, 'Without Helmet', num_images=5)


# **Object Detection**


1. We have developed a function, **detect_and_count_persons**, that leverages the YOLOv5 model to detect and create bounding boxes around all identified persons in the image. This function returns the bounding boxes along with the confidence scores for each detection.

2. The results from YOLOv5 are then passed to another function, **crop_person_images**. This function extracts and crops individual images of persons based on the bounding boxes provided by YOLO. We have set a confidence threshold of 0.70, meaning that only detections with a confidence score above 0.70 are considered valid. This helps eliminate false positives, ensuring only reliable detections are used for further processing.

3. Once we have the cropped images, we apply several augmentation techniques to increase the diversity of the dataset and make the model more robust.

In [ ]:
#Load the yolov5 model
model = torch.hub.load('ultralytics/yolov5', 'yolov5s')


In [ ]:
def detect_and_count_persons(model, image_path, show_image=False):
    """
    Function to detect and count the number of persons in an image.
    - Displays the original image.
    - Prints the count of persons detected.
    - Shows the image with bounding boxes drawn.

    Args:
        image_path (str): Path to the input image.
    """
    # Perform inference
    results = model(image_path)

    # Extract predictions
    detections = results.xyxy[0]  # Get bounding boxes in xyxy format
    person_count = 0

    # Iterate over detections and count 'person' labels (class 0 in COCO)
    for detection in detections:
        class_id = int(detection[5])  # Class ID is in the 6th position (zero-indexed)
        if class_id == 0 and 0.60 < detection[4]:  # 'person' class in COCO
            person_count += 1

    # Print the count of persons detected
    print(f"Number of persons detected: {person_count}")

    # Display the original image
    if show_image:
      original_image = Image.open(image_path)
      plt.figure(figsize=(10, 8))
      plt.imshow(original_image)
      plt.axis('off')
      plt.title('Original Image')
      plt.show()

      # Display the image with bounding boxes
      print("Displaying image with bounding boxes:")
      results.show()  # Shows images with bounding box

    return results

In [ ]:
# detect people in an image and plot the image with the corresponding bounding boxes
img_path = os.path.join(with_helmet_path_training, os.listdir(with_helmet_path_training)[1])
results = detect_and_count_persons(model, img_path, show_image= True)

In [ ]:
img_path_2 = os.path.join(without_helmet_path_training, os.listdir(without_helmet_path_training)[1])
results_2 = detect_and_count_persons(model, img_path_2, True)


In [ ]:
def crop_person_images(image_path, results, confidence_threshold=0.60):
    """
    Crops images of persons detected by YOLOv5.
    Only crops persons with a confidence score greater than the threshold.

    Args:
        image_path (str): Path to the input image.
        results (YOLO result object): The result object from YOLO inference.
        confidence_threshold (float): Minimum confidence value for cropping persons. Default is 0.80.

    Returns:
        List of PIL Image: List of cropped images of persons detected.
    """
    # Open the original image
    original_image = Image.open(image_path)

    # Extract predictions (xyxy format: [x1, y1, x2, y2, confidence, class_id])
    detections = results.xyxy[0]
    cropped_images = []

    # Iterate over detections and crop images for 'person' class with high confidence
    for detection in detections:
        x1, y1, x2, y2, confidence, class_id = detection.tolist()
        class_id = int(class_id)

        if class_id == 0 and confidence > confidence_threshold:
            # Crop the image based on the bounding box coordinates
            cropped_image = original_image.crop((x1, y1, x2, y2))
            cropped_images.append(cropped_image)

    return cropped_images


# **Dataset Creation for Image Classification**






1. For the helmet detection task, we begin by storing the cropped images of individuals, which are extracted from the original image using the YOLO algorithm. Along with each cropped image, we also store the corresponding label indicating whether the person is wearing a helmet or not. This label serves as the ground truth for training the model.

2. To ensure consistency in the dataset, we preprocess the cropped images by resizing them to the same dimensions, typically the input size required by the model (e.g., 224x224 for ResNet) and rescaling it by dividing the pixel values by 255. This step ensures that all input images have uniform size, which is crucial for feeding them into the neural network.

In [ ]:
# Creating the dataset from the images -
def dataset_generator(with_helmet_path, without_helmet_path):

  dataset = []

  all_helmet_images_name = os.listdir(with_helmet_path)
  all_non_helmet_images_name = os.listdir(without_helmet_path)

  for helmet_image_name in all_helmet_images_name:
    image_path = os.path.join(with_helmet_path, helmet_image_name)
    results = detect_and_count_persons(model, image_path=image_path)
    cropped_images = crop_person_images(image_path, results)

    for croped_image in cropped_images:
      dataset.append([croped_image, 1])

  for non_helmet_image_name in all_non_helmet_images_name:
    img_path = os.path.join(without_helmet_path, non_helmet_image_name)
    original_image = Image.open(img_path)
    dataset.append([original_image, 0])

  return dataset


In [ ]:
train_dataset = dataset_generator(with_helmet_path_training,without_helmet_path_training)
validation_dataset = dataset_generator(with_helmet_path_validation, without_helmet_path_validation)

In [ ]:
# A function to preprocess the images

def transform_image(image):
    img = image.convert('RGB')
    img = img.resize((224, 224))

    # Convert image to numpy array
    img = np.array(img) / 255.0
    return img


In [ ]:
def process_images_and_labels(data):
    images = []
    labels = []

    for image, label in data:
        img = transform_image(image)
        images.append(img)
        labels.append(label)

    images = np.array(images)
    labels = np.array(labels)

    return images, labels

In [ ]:
train_processed_images, train_labels = process_images_and_labels(train_dataset)
validation_processed_images, validation_labels = process_images_and_labels(validation_dataset)


In [ ]:
#spliting the training dataset into train and test
X_train, X_test, y_train, y_test = train_test_split(train_processed_images, train_labels, test_size=0.1, random_state=42, stratify=train_labels)
X_val, y_val = validation_processed_images, validation_labels


# **Image Classification**

In this step, we will load a pretrained ResNet50 model that has already been trained on a large and diverse dataset, such as ImageNet.

- ResNet50 is a deep convolutional neural network known for its residual connections, which allow it to efficiently train very deep networks without the problem of vanishing gradients.
- This architecture is widely used for tasks such as image classification, and it can be fine-tuned to recognize specific objects, like helmets, in our case.

## Utility Function

In [ ]:
# defining a function to compute different metrics to check performance of a classification model built using statsmodels
from sklearn.metrics import confusion_matrix, f1_score,accuracy_score, recall_score, precision_score, classification_report
def model_performance_classification(model, predictors, target):
    """
    Function to compute different metrics to check classification model performance

    model: classifier
    predictors: independent variables
    target: dependent variable
    """

    # checking which probabilities are greater than threshold
    predictions = np.round(model.predict(predictors)).flatten()

    # Calculate precision and recall
    accuracy = accuracy_score(target, predictions)
    precision = precision_score(target, predictions)
    recall = recall_score(target, predictions)
    f1 = f1_score(target, predictions)  # to compute F1-score

    # creating a dataframe of metrics
    df_perf = pd.DataFrame(
        {"Accuracy": accuracy, "Recall": recall, "Precision": precision, "F1 Score": f1,},
        index=[0],
    )

    return df_perf

## Model 1 ( Base model + output layer)

In [ ]:
# Load the Pretrained ResNet model
base_model = tf.keras.applications.ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
for layer in base_model.layers[:-10]:
    layer.trainable = False


In [ ]:
model = tf.keras.models.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# 7. Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


In [ ]:
train_datagen = ImageDataGenerator()

In [ ]:
#Train the model
epochs = 10
# Batch size
batch_size = 32

history_model_1 = model.fit(train_datagen.flow(X_train,y_train,
                                       shuffle= False),
                    epochs= epochs,
                    batch_size = batch_size,
                    validation_data=(X_val,y_val),
                    verbose=1)

Evaluating Model performance on training and validation data.



In [ ]:
model_perf_df_val = model_performance_classification(model, X_val, y_val)
model_perf_df_val

In [ ]:
model_perf_df_train = model_performance_classification(model, X_train, y_train)
model_perf_df_train


lets plot the training accuracy and validation accuracy against the epochs.

In [ ]:
plt.plot(history_model_1.history['accuracy'])
plt.plot(history_model_1.history['val_accuracy'])
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

# **Image Classification Performance Improvement and Final Model Selection**

## Model 2: (Base model + FFN)

Lets add a Feed forward neural network with 2 hidden layers. We will be increasing the learning rate while keeping the epochs same.

In [ ]:
base_model = tf.keras.applications.ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
for layer in base_model.layers[:-10]:
    layer.trainable = False


In [ ]:
model_FFN = tf.keras.models.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])
model_FFN.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


In [ ]:
epochs = 10
batch_size = 32

history_model_2 = model_FFN.fit(train_datagen.flow(X_train, y_train,
                                       batch_size=batch_size,
                                       seed=42,
                                       shuffle=False),
                    epochs=epochs,
                    steps_per_epoch=len(X_train) // batch_size,
                    validation_data=(X_val, y_val))


In [ ]:
model_FFN_perf_df_val = model_performance_classification(model_FFN, X_val, y_val)
model_FFN_perf_df_val

In [ ]:
model_FFN_perf_df_train = model_performance_classification(model_FFN, X_train, y_train)
model_FFN_perf_df_train


In [ ]:
plt.plot(history_model_2.history['accuracy'])
plt.plot(history_model_2.history['val_accuracy'])
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

## Model 3 (Base model + FFN + Data Augmentation) with dropout

Lets add a dropout and data augmentation. We will also decrease the learning rate so that we can reach a global minimnum.

In [ ]:
base_model = tf.keras.applications.ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
for layer in base_model.layers[:-10]:
    layer.trainable = False


In [ ]:
model_FFN_DA = tf.keras.models.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])
model_FFN_DA.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


In [ ]:
# Applying data augmentation
train_datagen = ImageDataGenerator(
                              rotation_range=__,
                              fill_mode='nearest',width_shift_range=__,height_shift_range=__,shear_range=0.3,zoom_range=0.4
                              )  # Complete the code to add Data augmentation by rotating the images 20 degrees and adding 20% width and height.

In [ ]:
epochs = 20
batch_size = 32

history_model_3 = model_FFN_DA.fit(train_datagen.flow(X_train, y_train,
                                       batch_size=batch_size,
                                       seed=42,
                                       shuffle=False),
                    epochs=epochs,
                    steps_per_epoch=len(X_train) // batch_size,
                    validation_data=(X_val, y_val))


In [ ]:
model_FFN_DA_perf_df_val = model_performance_classification(model_FFN_DA, X_val, y_val)
model_FFN_DA_perf_df_val

In [ ]:
model_FFN_DA_perf_df_train = model_performance_classification(model_FFN_DA, X_train, y_train)
model_FFN_DA_perf_df_train


In [ ]:
plt.plot(history_model_3.history['accuracy'])
plt.plot(history_model_3.history['val_accuracy'])
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

## Model 4 (Using a different optimizer and reducing batch size)

We will be using the same model architecture as above, but this time we will use Stochastic Gradient Descent optimizer with a larger learning rate to compile our model. We will decrease our batch size so that we can have faster convergences and better generalization.

In [ ]:
base_model = tf.keras.applications.ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
for layer in base_model.layers[:-10]:
    layer.trainable = False


In [ ]:
model_FFN_DA_2 = tf.keras.models.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001)
model_FFN_DA_2.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])


In [ ]:
# Applying data augmentation
train_datagen = ImageDataGenerator(
                              rotation_range=20,
                              fill_mode='nearest',width_shift_range=0.2,height_shift_range=0.2,shear_range=0.3,zoom_range=0.4
                              )


In [ ]:
model_FFN_DA_2_perf_df_val = model_performance_classification(model_FFN_DA_2, X_val, y_val)
model_FFN_DA_2_perf_df_val

In [ ]:
model_FFN_DA_2_perf_df_train = model_performance_classification(model_FFN_DA_2, X_train, y_train)
model_FFN_DA_2_perf_df_train


In [ ]:
plt.plot(history_model_4.history['accuracy'])
plt.plot(history_model_4.history['val_accuracy'])
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

## Model Performance Comparison and Final Model Selection

In [ ]:
models_train_comp_df = pd.concat(
    [
        model_perf_df_train.T,
        model_FFN_perf_df_train.T,
        model_FFN_DA_perf_df_train.T,
        model_FFN_DA_2_perf_df_train.T,


    ],
    axis=1,
)
models_train_comp_df.columns = [
    "Model 1: Simple",
    "Model 2: FFN",
    "Model 3: DA",
    'Model 4: SGD'
]

models_val_comp_df = pd.concat(   # Complete the code to concatenate the validation performances of all the models.
    [
        ______.T,
        ______.T,
        ______.T,
        ______.T

    ],
    axis=1,
)
models_val_comp_df.columns = [
    "Model 1: Simple",
    "Model 2: FFN",
    "Model 3: DA",
    'Model 4: SGD'
]



In [ ]:
print('PERFORMANCE OF MODELS ON THE TRAINING SET')
models_train_comp_df

In [ ]:
print('PERFORMANCE OF MODELS ON VALIDATION SET')
models_val_comp_df

## Model Performance Check on Test Set

In [ ]:
best_model = model_FFN_DA_2  # Assuming this performed best empirically.


In [ ]:
predictions = np.round(best_model.predict(X_test))
print(classification_report(y_test, predictions))


Lets plot the confusion matrix to gain a deeper insight.

In [ ]:
#Compute Confusion Matrix
cm = confusion_matrix(y_test, predictions)

# Confusion Matrix Labels
labels = ['Class 0 (No Helmet)', 'Class 1 (Helmet)']

# Check the content of the confusion matrix to ensure it's correct
print('Confusion Matrix:')

# Plot the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels,
            cbar=False, linewidths=1, linecolor='black',
            square=True)

# Add Titles and Axis Labels
plt.title('Confusion Matrix', fontsize=16)
plt.xlabel('Predicted Labels', fontsize=12)
plt.ylabel('True Labels', fontsize=12)

# # Ensure the plot window is shown
plt.show()


Lets plot some images of the test dataset along with the predictions.

In [ ]:
num_images = 5

selected_images_indices = random.sample(range(len(X_test)), num_images)
selected_images = X_test[selected_images_indices]

class_dict = {0: 'No Helmet Detected', 1: 'Helmet Detected'}

# Plot the images
plt.figure(figsize=(15, 5))
for i, image in enumerate(selected_images):
    confidence = best_model.predict(np.expand_dims(image, axis=0),verbose=2)
    prediction = int(np.round(confidence))
    if prediction == 0:
        confidence = (1 - confidence) * 100
    else:
        confidence = confidence[0][0] * 100
    detected_class = class_dict[prediction]
    plt.subplot(1, num_images, i + 1)  # Arrange images in 1 row
    plt.imshow(image)
    plt.axis('off')  # Hide axes
    plt.title(f' image #{i + 1} \nPredicted Class:{detected_class}\nConfidence:{np.round(confidence,2)}%', fontsize= 10)
plt.tight_layout()
plt.show()


# **Actionable Insights and Recommendations**


## Insights

-


## Recommendations

-


<font size=6 color='blue'>Power Ahead!</font>
___